In [67]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.naive_bayes import MultinomialNB

import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 123 # constant for reproducibility

In [68]:
# repeat steps in EDA to format dataframe correctly
netflix_df = pd.read_csv('netflix_user_behavior_dataset.csv')

netflix_df['user_id'] = netflix_df['user_id'].astype(str)
netflix_df['age'] = netflix_df['age'].astype(int)
netflix_df['gender'] = netflix_df['gender'].astype('category')
netflix_df['country'] = netflix_df['country'].astype('category')
netflix_df['account_age_months'] = netflix_df['account_age_months'].astype(int)
netflix_df['subscription_type'] = netflix_df['subscription_type'].astype('category')
netflix_df['monthly_fee'] = netflix_df['monthly_fee'].astype(float)
netflix_df['payment_method'] = netflix_df['payment_method'].astype('category')
netflix_df['primary_device'] = netflix_df['primary_device'].astype('category')
netflix_df['devices_used'] = netflix_df['devices_used'].astype(int)
netflix_df['favorite_genre'] = netflix_df['favorite_genre'].astype('category')
netflix_df['avg_watch_time_minutes'] = netflix_df['avg_watch_time_minutes'].astype(int)
netflix_df['watch_sessions_per_week'] = netflix_df['watch_sessions_per_week'].astype(int)
netflix_df['binge_watch_sessions'] = netflix_df['binge_watch_sessions'].astype(int)
netflix_df['completion_rate'] = netflix_df['completion_rate'].astype(int)
netflix_df['rating_given'] = netflix_df['rating_given'].astype(float)
netflix_df['content_interactions'] = netflix_df['content_interactions'].astype(int)
netflix_df['recommendation_click_rate'] = netflix_df['recommendation_click_rate'].astype(int)
netflix_df['days_since_last_login'] = netflix_df['days_since_last_login'].astype(int)

churn_map = {'No': False, 'Yes': True}
netflix_df['churned'] = netflix_df['churned'].map(churn_map).astype(bool)

In [69]:
# separate into train and test
predictors = pd.get_dummies(netflix_df.drop(columns=['user_id', 'churned']), drop_first=True)
target = netflix_df['churned']
X_train, X_test, y_train, y_test = train_test_split(predictors, target, test_size=0.2, random_state=RANDOM_STATE) # use random state constant for reproducibility

Categorical variables transformed to ensure compatibility with machine learning algorithms that require numeric input.

In [70]:
# Baseline Model - Logistic Regression
log_model_bal = LogisticRegression(max_iter=5000)

# train
log_model_bal.fit(X_train, y_train)

# predict
log_bal_preds = log_model_bal.predict(X_test)

In [71]:
print(confusion_matrix(y_test, log_bal_preds))
print(classification_report(y_test, log_bal_preds))

[[8010    0]
 [1990    0]]
              precision    recall  f1-score   support

       False       0.80      1.00      0.89      8010
        True       0.00      0.00      0.00      1990

    accuracy                           0.80     10000
   macro avg       0.40      0.50      0.44     10000
weighted avg       0.64      0.80      0.71     10000



In [72]:
# Baseline Model - Logistic Regression
log_model_bal = LogisticRegression(max_iter=5000, class_weight='balanced')

# train
log_model_bal.fit(X_train, y_train)

# predict
log_bal_preds = log_model_bal.predict(X_test)

In [73]:
print(confusion_matrix(y_test, log_bal_preds))
print(classification_report(y_test, log_bal_preds))

[[4084 3926]
 [1007  983]]
              precision    recall  f1-score   support

       False       0.80      0.51      0.62      8010
        True       0.20      0.49      0.28      1990

    accuracy                           0.51     10000
   macro avg       0.50      0.50      0.45     10000
weighted avg       0.68      0.51      0.56     10000



When using logistic regression, the model initially predicted all users as non-churn, resulting in an accuracy of 80% but failing to identify any churned users. After applying class weighting, the model improved its ability to detect churn, with recall increasing from 0.00 to approximately 0.49. However, this improvement came at the cost of lower precision and overall accuracy, indicating that the model produced more false positives while better identifying churned users.

In [74]:
tree_model = DecisionTreeClassifier(random_state=RANDOM_STATE)
tree_model.fit(X_train, y_train)

tree_preds = tree_model.predict(X_test)

In [75]:
print(confusion_matrix(y_test, tree_preds))
print(classification_report(y_test, tree_preds))

[[6265 1745]
 [1531  459]]
              precision    recall  f1-score   support

       False       0.80      0.78      0.79      8010
        True       0.21      0.23      0.22      1990

    accuracy                           0.67     10000
   macro avg       0.51      0.51      0.51     10000
weighted avg       0.69      0.67      0.68     10000



The Decision Tree model achieved higher overall accuracy (67%) compared to logistic regression. However, when identifying churned users, its recall was lower than the logistic regression model, correctly identifying only 23% of churned users. The class-weighted logistic regression model detected churn with a higher recall measure of near 49%, but sacrifices accuracy. This conveys the tradeoffs between the ability to correctly identify churned customers and maximizing model accuracy.

# NOTES

PCA
Balance train/test
Naive Bayes Classifier
Metric calculations
Logistic regression using smtools library
Clustering analysis
C5.0, C4.5, CART trees

In [76]:
# PCA   
from sklearn.decomposition import PCA
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, classification_report

In [77]:
categorical_cols = predictors.select_dtypes(include=['category']).columns
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ],
    remainder='passthrough'
)
X_processed = preprocessor.fit_transform(predictors)

In [78]:
# reduce features to 5 principal components
pca = PCA(n_components=5)
X_pca = pca.fit_transform(X_processed)

In [79]:
# split PCA data
X_train_pca, X_test_pca, y_train_pca, y_test_pca = train_test_split(
    X_pca, target, test_size=0.2, random_state=123
)

In [80]:
# Logistic Regression with PCA
pca_model = LogisticRegression(max_iter=1000, class_weight='balanced')
pca_model.fit(X_train_pca, y_train_pca)
y_pred_pca = pca_model.predict(X_test_pca)

In [81]:
print("Logistic Regression with PCA:")
print("Accuracy:", accuracy_score(y_test_pca, y_pred_pca))
print(classification_report(y_test_pca, y_pred_pca, zero_division=0))

Logistic Regression with PCA:
Accuracy: 0.4994
              precision    recall  f1-score   support

       False       0.80      0.50      0.62      8010
        True       0.20      0.49      0.28      1990

    accuracy                           0.50     10000
   macro avg       0.50      0.50      0.45     10000
weighted avg       0.68      0.50      0.55     10000



For PCA, the data was reduced to 5 principal components, and a Logistic Regression model was trained on the transformed dataset. The results showed that PCA did not significantly improve model performance. Accuracy remained around 50%, and the model’s ability to detect churn cases was similar to the original Logistic Regression model without PCA.

This suggests that dimensionality reduction did not enhance the model’s ability to distinguish between churn and non-churn users. Overall, PCA did not provide a clear benefit for this dataset.

In [82]:
!pip install -U imbalanced-learn scikit-learn

In [83]:
# SMOTE
from imblearn.over_sampling import SMOTE

In [84]:
smote = SMOTE(random_state=123)
X_resampled, y_resampled = smote.fit_resample(X_processed, target)

In [85]:
# check new class distribution
print(y_resampled.value_counts())

churned
False    40036
True     40036
Name: count, dtype: int64


In [86]:
# train/test split on balanced data
from sklearn.model_selection import train_test_split

X_train_sm, X_test_sm, y_train_sm, y_test_sm = train_test_split(
    X_resampled, y_resampled, test_size=0.2, random_state=123
)

In [87]:
# test model
smote_model = LogisticRegression(max_iter=1000)
smote_model.fit(X_train_sm, y_train_sm)
y_pred_sm = smote_model.predict(X_test_sm)

print("Logistic Regression with SMOTE:")
print("Accuracy:", accuracy_score(y_test_sm, y_pred_sm))
print(classification_report(y_test_sm, y_pred_sm, zero_division=0))

Logistic Regression with SMOTE:
Accuracy: 0.5107711520449578
              precision    recall  f1-score   support

       False       0.51      0.50      0.51      8033
        True       0.51      0.52      0.52      7982

    accuracy                           0.51     16015
   macro avg       0.51      0.51      0.51     16015
weighted avg       0.51      0.51      0.51     16015



In [88]:
# Naive Bayes
from sklearn.naive_bayes import GaussianNB

In [89]:
nb_model = GaussianNB()
nb_model.fit(X_train_sm, y_train_sm)
y_pred_nb = nb_model.predict(X_test_sm)

In [90]:
print("Naive Bayes Results:")
print("Accuracy:", accuracy_score(y_test_sm, y_pred_nb))
print(classification_report(y_test_sm, y_pred_nb, zero_division=0))

Naive Bayes Results:
Accuracy: 0.7495472994068061
              precision    recall  f1-score   support

       False       0.72      0.83      0.77      8033
        True       0.79      0.67      0.73      7982

    accuracy                           0.75     16015
   macro avg       0.76      0.75      0.75     16015
weighted avg       0.76      0.75      0.75     16015



In [91]:
# CART
dt_model = DecisionTreeClassifier(
    criterion='entropy',  # <-- THIS is key
    random_state=123
)
dt_model.fit(X_train_sm, y_train_sm)
y_pred_dt = dt_model.predict(X_test_sm)

In [92]:
print("Decision Tree (Entropy / C4.5-style) Results:")
print("Accuracy:", accuracy_score(y_test_sm, y_pred_dt))
print(classification_report(y_test_sm, y_pred_dt, zero_division=0))

Decision Tree (Entropy / C4.5-style) Results:
Accuracy: 0.794442709959413
              precision    recall  f1-score   support

       False       0.80      0.78      0.79      8033
        True       0.79      0.81      0.80      7982

    accuracy                           0.79     16015
   macro avg       0.79      0.79      0.79     16015
weighted avg       0.79      0.79      0.79     16015



In [93]:
# statsmodels logistic regression
import statsmodels.api as sm
import numpy as np

In [94]:
# convert to numeric float array
X_dense = np.asarray(X_processed).astype(float)

In [95]:
# add intercept
X_sm = sm.add_constant(X_dense)

In [96]:
# make sure target is numeric
y_sm = target.astype(int)

In [97]:
# fit model
model_sm = sm.Logit(y_sm, X_sm)
result = model_sm.fit()

Optimization terminated successfully.
         Current function value: 0.499003
         Iterations 5


In [98]:
print(result.summary())

                           Logit Regression Results                           
Dep. Variable:                churned   No. Observations:                50000
Model:                          Logit   Df Residuals:                    49961
Method:                           MLE   Df Model:                           38
Date:                Sat, 04 Apr 2026   Pseudo R-squ.:               0.0008009
Time:                        20:06:52   Log-Likelihood:                -24950.
converged:                       True   LL-Null:                       -24970.
Covariance Type:            nonrobust   LLR p-value:                    0.3815
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const         -1.2357      0.108    -11.404      0.000      -1.448      -1.023
x1            -0.0005      0.001     -0.547      0.584      -0.002       0.001
x2            -0.0010      0.001     -1.573      0.1

In [99]:
# clustering/k-means
from sklearn.cluster import KMeans

In [100]:
kmeans = KMeans(n_clusters=3, random_state=123)
clusters = kmeans.fit_predict(X_processed)

In [101]:
print("Cluster assignments (first 20 users):")
print(clusters[:20])

Cluster assignments (first 20 users):
[0 1 0 0 0 1 2 0 2 0 1 2 1 1 2 0 2 2 0 1]


In [102]:
netflix_df['cluster'] = clusters

In [103]:
print(netflix_df.groupby('cluster').mean(numeric_only=True))

               age  account_age_months  monthly_fee  devices_used  \
cluster                                                             
0        41.028238           29.782761    12.344808      2.012576   
1        40.981510           30.077012    12.330453      1.983585   
2        40.927727           29.767248    12.294786      2.000836   

         avg_watch_time_minutes  watch_sessions_per_week  \
cluster                                                    
0                    250.980720                 9.978822   
1                     57.337707                 9.998352   
2                    153.792731                 9.983886   

         binge_watch_sessions  completion_rate  rating_given  \
cluster                                                        
0                    7.026636        64.514445      3.013561   
1                    7.014890        64.556905      3.000781   
2                    6.966161        64.529542      2.992504   

         content_interactions  r